# Qwen3.8-27B · 64K · Dual Tesla T4 + Vision — No Kaggle Secrets

Runs the original Q4_K_M model with tensor-split GPU inference, GPU vision, and MTP speculative decoding. Preserves the original **fixed 65,536-token configuration**; it has **not been validated on this Kaggle hardware at 64K**.

**No Kaggle Secrets or Cloudflare account required:**
- The GGUFs in public `ggml-org/Qwen3.8-27B-GGUF` download anonymously (or are reused from existing Kaggle files).
- A fresh random API key is generated inside Python on startup; no API key needs to be supplied. Keep it private.
- Cloudflare **Quick Tunnel** automatically creates a new `https://<random>.trycloudflare.com/v1` URL. No `CF_TUNNEL_TOKEN`, account, DNS, or custom domain.

**Kaggle:** Enable Internet, select **GPU T4 × 2**, and run all cells in order. The notebook's final cell starts a non-blocking watchdog. The API and tunnel exist only while the Kaggle session and processes run. A renewed tunnel has a **different URL**.

**Limitations:** Cloudflare Quick Tunnels are intended for development, have no uptime guarantee, and **do not support Server-Sent Events (SSE)**. For public API calls, use `"stream": false` in `/v1/chat/completions`. Keep the generated API key secret and do not publish cell output containing it. Long requests may still run into network/proxy timeouts. The 64K context may fail to fit in dual T4 VRAM (no automatic downgrade).

## 1 · Configuration (no stored tokens or Kaggle Secrets)

In [ ]:
from pathlib import Path
import os, re, shlex, shutil, signal, subprocess, sys, time, json, threading, secrets
import requests

# Verified v15 inference parameters; 64K context newly requested (not yet GPU-validated). No benchmarks or candidate switching.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
# Never send a cached Hugging Face credential implicitly for this public repo.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
WORK = Path("/kaggle/working/qwen38_official_dflash_v9")  # Reuse existing model files and binaries.
BIN, LOG_DIR, STATE_DIR, MODEL_DIR = (WORK/x for x in ("bin", "logs", "state", "models"))
for folder in (WORK, BIN, LOG_DIR, STATE_DIR, MODEL_DIR):
    folder.mkdir(parents=True,exist_ok=True)
SERVER, CLOUDFLARED = BIN/'llama-server', BIN/'cloudflared'
BASE_REPO = "ggml-org/Qwen3.8-27B-GGUF"
MODEL_FILE = "Qwen3.8-27B-Q4_K_M.gguf"
MTP_FILE = "mtp-Qwen3.8-27B-Q4_0.gguf"
MMPROJ_FILE = "mmproj-Qwen3.8-27B-Q8_0.gguf"
ALIAS = "qwen3.8-27b-official"
PORT, LLAMA_PORT = 8080, 8081
PUBLIC_BASE_URL = None  # Assigned when Cloudflare creates a new Quick Tunnel.
MAX_OUTPUT_GUARD = 8192

# FIXED settings supported by the uploaded measured run.
PRODUCTION = {
    'name':'FINAL_64K_tensor_mtp2_p040_vision',
    'split':'tensor', 'tensor_split':'1,1', 'ctx':65536,
    'kv':'f16', 'batch':2048, 'ubatch':256,
    'spec_kind':'mtp', 'n':2, 'pmin':0.40,
    'vision':True, 'image_min':1024, 'image_max':1024,
}
ENABLE_VISION=True
VISION_REQUIRED=True
STORAGE_PREFER_RAM_FOR_DFLASH=False
STORAGE_ALLOW_MEMFD=True
STORAGE_RAM_RESERVE_GIB=8.0
STORAGE_DISK_RESERVE_MIB=256
FORCE_REBUILD=False
TRY_NCCL=True
LLAMA_CPP_REF='master'  # Used only when a suitable restored llama-server binary is unavailable.

# A fresh key protects the temporary public endpoint; no Kaggle secret setup is needed.
# A rerun of this cell in the same kernel retains the key so existing clients still work.
if not globals().get('_AUTO_API_KEY'):
    _AUTO_API_KEY = secrets.token_urlsafe(32)
LLAMA_API_KEY = _AUTO_API_KEY
RUN_PUBLIC_TUNNEL = True  # Cloudflare Quick Tunnel; no account, domain or token.
print('Serving:', ALIAS, '| vision:', PRODUCTION['vision'], '| context:', PRODUCTION['ctx'])
print('Cloudflare Quick Tunnel:', 'enabled' if RUN_PUBLIC_TUNNEL else 'local-only')
print('Client API key: generated automatically in this session (not read from Secrets).')

## 2 · Reuse/download validated model weights and projector
Downloads only the base model, MTP draft, and projector. Reuses saved weights from previous Kaggle runs or Kaggle Input datasets, including scratch storage for smaller files.

In [ ]:
# v9.2: Storage-aware GGUF loading with zero-staging streaming for optional optional GGUF.
# Never deletes or replaces complete models in /kaggle/working.
import subprocess, shutil, os, pathlib, re, hashlib, time
import requests

subprocess.run(['apt-get','update','-qq'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['apt-get','install','-y','-qq','git','cmake','ninja-build','build-essential','wget','curl'],
               check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['python','-m','pip','install','-q','huggingface_hub>=1.3,<2.0','hf_xet',
                'fastapi','uvicorn','httpx','Pillow','pandas'], check=True)
if TRY_NCCL:
    subprocess.run('apt-get install -y -qq libnccl2', shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

from huggingface_hub import hf_hub_download, get_hf_file_metadata, hf_hub_url
INPUT_ROOT = pathlib.Path('/kaggle/input')
GIB = 1024**3
MIB = 1024**2

# Do not overwrite the registry when rerunning the cell in the same kernel.
# Holding the FD open is ESSENTIAL for llama-server to read /proc/<kernel_pid>/fd/<N>.
if 'RAM_MODEL_FDS' not in globals():
    RAM_MODEL_FDS = {}

def _human(n): return f'{n/GIB:.2f} GiB'

def _free(folder):
    try: return shutil.disk_usage(folder).free
    except Exception: return 0

def _ram_available():
    available = None
    try:
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                available = int(line.split()[1]) * 1024
                break
    except Exception: pass
    # Container limit can be smaller than system MemAvailable.
    cg = pathlib.Path('/sys/fs/cgroup')
    try:
        mx, used = cg/'memory.max', cg/'memory.current'
        if mx.exists() and used.exists():
            limit = mx.read_text().strip()
            if limit != 'max':
                left = max(0,int(limit)-int(used.read_text().strip()))
                available = min(left,available) if available is not None else left
        else:
            mx, used = cg/'memory/memory.limit_in_bytes', cg/'memory/memory.usage_in_bytes'
            if mx.exists() and used.exists():
                limit = int(mx.read_text().strip())
                if limit < 2**60:
                    left = max(0,limit-int(used.read_text().strip()))
                    available = min(left,available) if available is not None else left
    except Exception as exc:
        print('Note: unable to read cgroup quota:',str(exc)[:90])
    return available

def _ram_safe(required):
    avail = _ram_available()
    needed = required + int(STORAGE_RAM_RESERVE_GIB * GIB)
    if avail is None:
        print('Cannot verify available system/container RAM; RAM download disabled')
        return False
    if avail < needed:
        print('RAM guard: need',_human(needed),'including reserve; available',_human(avail))
        return False
    return True

def _input_file(name):
    if not INPUT_ROOT.exists(): return None
    for found in INPUT_ROOT.rglob(name):
        if found.is_file(): return found
    return None

for binary in ('llama-server','llama-bench','cloudflared'):
    target = BIN/binary
    source = _input_file(binary)
    if source and not target.exists():
        shutil.copy2(source,target)
        print('Restored binary:',binary)
    if target.exists(): target.chmod(0o755)

def _remote_info(repo,filename):
    try:
        m = get_hf_file_metadata(hf_hub_url(repo,filename), token=False)
        size = int(m.size) if m.size else None
        etag = (m.etag or '').strip('"')
        sha = etag.lower() if re.fullmatch(r'[a-fA-F0-9]{64}',etag) else None
        return (size,sha)
    except Exception as exc:
        print('Remote metadata unavailable:',filename,str(exc)[:150])
        return (None,None)

def _good(path,minimum_gb,expected):
    if path is None: return False
    try:
        p = pathlib.Path(path)
        if not p.is_file(): return False
        if not (p.stat().st_size == expected if expected is not None
                else p.stat().st_size > minimum_gb * 1e9): return False
        with open(p,'rb') as f: return f.read(4) == b'GGUF'
    except Exception: return False

def _existing(filename,minimum_gb,expected):
    # Check prior RAM file, existing user data, then in-progress notebook paths.
    p = RAM_MODEL_FDS.get(filename,{}).get('path')
    if _good(p,minimum_gb,expected): return pathlib.Path(p)
    p = _input_file(filename)
    if _good(p,minimum_gb,expected):
        print('Reusing read-only Kaggle Input model:',p)
        return p
    for root in (MODEL_DIR, pathlib.Path('/kaggle/temp/qwen38_dflash_v92/models'),
                 pathlib.Path('/dev/shm/qwen38_dflash_v92/models'),
                 pathlib.Path('/kaggle/temp/qwen38_dflash_v91/models'),
                 pathlib.Path('/dev/shm/qwen38_dflash_v91/models')):
        p = root/filename
        if _good(p,minimum_gb,expected):
            print('Reusing existing completed GGUF:',p)
            return p
    return None

# Returns (method, destination dir); memfd is used only for the optional optional GGUF draft.
def _storage_choice(expected,optional,filename):
    extra = 128*MIB if optional else 256*MIB
    required = (expected+extra) if expected is not None else (2*GIB if optional else 24*GIB)
    work_dev = MODEL_DIR.stat().st_dev
    def on_disk():
        if _free(MODEL_DIR) > required + STORAGE_DISK_RESERVE_MIB*MIB:
            return ('disk',MODEL_DIR)
        for root in ('/kaggle/temp','/tmp'):
            p=pathlib.Path(root)
            try:
                if p.is_dir() and os.access(p,os.W_OK) and p.stat().st_dev != work_dev and _free(p) > required:
                    dest=p/'qwen38_dflash_v92'/'models'
                    dest.mkdir(parents=True,exist_ok=True)
                    return ('scratch',dest)
            except Exception: pass
        return None
    def in_ram():
        if not optional: return None
        if not _ram_safe(required): return None
        shm = pathlib.Path('/dev/shm')
        try:
            if shm.is_dir() and os.access(shm,os.W_OK) and _free(shm) > required:
                dest=shm/'qwen38_dflash_v92'/'models'
                dest.mkdir(parents=True,exist_ok=True)
                return ('shm',dest)
        except Exception as exc:
            print('Shared memory path not available:',str(exc)[:100])
        if STORAGE_ALLOW_MEMFD and hasattr(os,'memfd_create') and pathlib.Path('/proc/self/fd').is_dir():
            return ('memfd',None)
        return None
    choice = (in_ram() or on_disk()) if STORAGE_PREFER_RAM_FOR_DFLASH and optional else (on_disk() or in_ram())
    if choice is None:
        print('No safe location for',filename,'| working free',_human(_free(MODEL_DIR)),
              '| /dev/shm free',_human(_free('/dev/shm')),'| RAM avail',
              _human(_ram_available()) if _ram_available() is not None else 'unknown')
    else:
        print('Storage strategy:',filename,'=>',choice[0],(str(choice[1]) if choice[1] else 'kernel RAM-backed memfd'))
    return choice

def _stream_from_hf(repo,filename,destination,expected,sha256_expected,method):
    # Raw HTTP streaming avoids a second HF/Xet reconstruction or disk staging copy.
    # This uses one RAM copy in /dev/shm, or one memfd outside the shm mount quota.
    if method == 'memfd':
        fd=os.memfd_create('qwen38-model-gguf',0)
        output=os.fdopen(fd,'wb',closefd=False)
        partial=None
    else:
        destination.mkdir(parents=True,exist_ok=True)
        final=destination/filename
        partial=destination/(filename+'.v92.download')
        fd=None
        output=open(partial,'wb')
    hasher=hashlib.sha256() if sha256_expected else None
    count=0
    try:
        url=hf_hub_url(repo,filename)
        headers={}  # Public repository: anonymous download
        with requests.get(url,headers=headers,stream=True,timeout=(25,180),allow_redirects=True) as response:
            response.raise_for_status()
            for piece in response.iter_content(chunk_size=4*MIB):
                if not piece: continue
                output.write(piece)
                count+=len(piece)
                if hasher: hasher.update(piece)
                if expected is not None and count>expected:
                    raise RuntimeError('Remote file larger than expected')
                if count//(256*MIB)!=(count-len(piece))//(256*MIB):
                    print(' ',filename, _human(count), 'downloaded')
        output.flush()
        if count < 4:
            raise RuntimeError('Downloaded content too short to be a GGUF')
        if method=='memfd':
            magic=os.pread(fd,4,0)
        else:
            with open(partial,'rb') as _check: magic=_check.read(4)
        if magic != b'GGUF':
            raise RuntimeError('Download did not contain GGUF model bytes')
        if expected is not None and count!=expected:
            raise RuntimeError(f'Incomplete streamed file: {count} bytes, expected {expected}')
        if hasher and hasher.hexdigest().lower()!=sha256_expected.lower():
            raise RuntimeError('SHA-256 mismatch for '+filename)
        if method == 'memfd':
            output.close()  # fd remains open because closefd=False
            path=pathlib.Path(f'/proc/{os.getpid()}/fd/{fd}')
            # A separate process must be able to open the path used by llama-server.
            test=subprocess.run(['python','-c',
                   'import sys; f=open(sys.argv[1],"rb"); assert f.read(4)==b"GGUF"',str(path)],
                   capture_output=True,text=True,timeout=15)
            if test.returncode:
                raise RuntimeError('Child process cannot open RAM fd: '+test.stderr[:180])
            RAM_MODEL_FDS[filename]={'fd':fd,'path':str(path),'size':count}
            print('Memory draft ready at',path,'(kernel must remain running)')
            return path
        output.close()
        os.replace(partial,final)
        return final
    except Exception:
        try: output.close()
        except Exception: pass
        if fd is not None:
            try: os.close(fd)
            except Exception: pass
        if partial is not None:
            partial.unlink(missing_ok=True)  # only our own incomplete v9.2 temp file
        raise

def _weight(repo,filename,min_gb,optional=False):
    expected,sha = _remote_info(repo,filename)
    existing = _existing(filename,min_gb,expected)
    if existing is not None: return existing
    choice = _storage_choice(expected,optional,filename)
    if choice is None:
        if optional:
            print('Skipping optional draft/projector:',filename,'(insufficient storage/RAM)')
            return None
        raise RuntimeError('Not enough disk for base model. Attach GGUF as a Kaggle Input Dataset; do not clear complete weights.')
    method, dest=choice
    try:
        # For optional weights, avoid the Xet cache + reconstruction disk duplication.
        # Non-optional base uses the regular resumable downloader.
        if optional:
            path=_stream_from_hf(repo,filename,dest,expected,sha,method)
        else:
            path=pathlib.Path(hf_hub_download(repo_id=repo,filename=filename,local_dir=str(dest),token=False))
        if not _good(path,min_gb,expected):
            raise RuntimeError('Downloaded GGUF did not validate: '+str(path))
        print('Validated',filename,'at',path,'size',_human(path.stat().st_size))
        return path
    except Exception as exc:
        if optional:
            print('OPTIONAL MODEL UNAVAILABLE:',filename,':',str(exc)[:350])
            print('Continuing without this optional draft; no complete model was deleted.')
            return None
        raise

MODEL_PATH = _weight(BASE_REPO, MODEL_FILE, 17.0)
MTP_PATH = _weight(BASE_REPO, MTP_FILE, 1.4, optional=True)
MMPROJ_PATH = _weight(BASE_REPO, MMPROJ_FILE, 0.55, optional=True)
if MTP_PATH is None:
    raise RuntimeError('Production MTP weights missing; refusing silent downgrade to plain inference.')
if MMPROJ_PATH is None:
    raise RuntimeError('Vision projector missing; refusing silent downgrade to text-only inference.')
print('Validated base, MTP and vision projector; ready to start production server.')

## 3 · Prepare CUDA llama.cpp (T4 sm_75)
Reuses a capable existing binary where possible. Compiles only if the required MTP, image, or tensor features are missing.

In [ ]:
def server_help():
    result=subprocess.run([str(SERVER),'--help'], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True,timeout=60)
    return result.stdout

def runtime_caps():
    if not SERVER.exists(): return {'ok':False}
    try: h=server_help()
    except Exception: return {'ok':False}
    required=['--split-mode','--tensor-split','--spec-draft-model',
              '--spec-draft-n-max','--spec-draft-p-min',
              'draft-mtp','--mmproj','--image-min-tokens','--image-max-tokens',
              '--flash-attn']
    return {'ok': all(flag in h for flag in required),
            'tensor':bool(re.search(r'split-mode[^\n]*tensor',h)) or '- tensor:' in h,
            'fit':'--fit' in h}

CAPS=runtime_caps()
print('Restored runtime:',CAPS)
if FORCE_REBUILD or not CAPS.get('ok') or not CAPS.get('tensor'):
    print('Compiling CUDA llama-server for NVIDIA T4 (sm_75).')
    src=Path('/kaggle/temp/llama.cpp-qwen38-prod')
    shutil.rmtree(src,ignore_errors=True)
    subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp',str(src)],check=True)
    if LLAMA_CPP_REF!='master':
        subprocess.run(['git','-C',str(src),'fetch','--depth','1','origin',LLAMA_CPP_REF],check=True)
        subprocess.run(['git','-C',str(src),'checkout','FETCH_HEAD'],check=True)
    build=src/'build'
    nccl_ok=False
    if TRY_NCCL:
        subprocess.run('apt-get install -y -qq libnccl2 libnccl-dev',shell=True,
                       stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        nccl_ok=Path('/usr/include/nccl.h').exists()
    def compile_server(use_nccl):
        shutil.rmtree(build,ignore_errors=True)
        args=['cmake','-S',str(src),'-B',str(build),'-G','Ninja',
              '-DBUILD_SHARED_LIBS=OFF','-DGGML_CUDA=ON','-DGGML_CUDA_NO_VMM=ON',
              '-DGGML_CUDA_FA=ON','-DGGML_CUDA_GRAPHS=ON',
              '-DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc',
              '-DCUDAToolkit_ROOT=/usr/local/cuda','-DCMAKE_BUILD_TYPE=Release',
              '-DCMAKE_CUDA_ARCHITECTURES=75','-DLLAMA_BUILD_TESTS=OFF',
              '-DLLAMA_BUILD_EXAMPLES=OFF']
        if use_nccl: args+=['-DGGML_CUDA_NCCL=ON']
        subprocess.run(args,check=True)
        subprocess.run(['cmake','--build',str(build),'--target','llama-server',
                        '-j',str(max(2,min(4,os.cpu_count() or 2)))],check=True)
    try: compile_server(nccl_ok)
    except subprocess.CalledProcessError:
        if not nccl_ok: raise
        print('NCCL compile failed; rebuilding without NCCL.')
        compile_server(False)
    shutil.copy2(build/'bin'/'llama-server',SERVER)
    SERVER.chmod(0o755)
    (WORK/'llama_cpp_commit.txt').write_text(
        subprocess.check_output(['git','-C',str(src),'rev-parse','HEAD'],text=True).strip()+'\n')
    CAPS=runtime_caps()
if not CAPS.get('ok') or not CAPS.get('tensor'):
    raise RuntimeError('llama-server binary does not support tensor split, MTP and vision flags.')
missing=subprocess.getoutput(f"ldd {shlex.quote(str(SERVER))} | grep 'not found' || true").strip()
if missing: raise RuntimeError('Unresolved shared libraries: '+missing)
print('CUDA server ready.')


## 4 · Production server launcher (fixed 64K profile)

In [ ]:
import requests
SERVER_LOG=LOG_DIR/'llama-server.log'
SERVER_PIDFILE=STATE_DIR/'llama-server.pid'
SERVER_PROC=None

def auth_headers():
    return {'Authorization':'Bearer '+LLAMA_API_KEY, 'Content-Type':'application/json'}
def backend_health():
    try:
        return requests.get(f'http://127.0.0.1:{LLAMA_PORT}/health',timeout=3).status_code==200
    except requests.RequestException: return False
def local_health():
    try:
        return requests.get(f'http://127.0.0.1:{PORT}/health',timeout=3).status_code==200
    except requests.RequestException: return False

def stop_pidfile(pidfile,timeout=8):
    if not pidfile.exists():return
    try: pid=int(pidfile.read_text().strip())
    except (ValueError,OSError): pidfile.unlink(missing_ok=True); return
    try:
        os.kill(pid,signal.SIGTERM)
        for _ in range(timeout*10):
            try:os.kill(pid,0)
            except ProcessLookupError:break
            time.sleep(.1)
        else: os.kill(pid,signal.SIGKILL)
    except ProcessLookupError: pass
    except PermissionError: pass
    pidfile.unlink(missing_ok=True)

def kill_server():
    global SERVER_PROC
    if SERVER_PROC is not None and SERVER_PROC.poll() is None:
        try:os.killpg(os.getpgid(SERVER_PROC.pid),signal.SIGKILL)
        except ProcessLookupError:pass
        try:SERVER_PROC.wait(timeout=8)
        except subprocess.TimeoutExpired:pass
    elif SERVER_PIDFILE.exists():
        try:
            pid=int(SERVER_PIDFILE.read_text().strip())
            os.killpg(os.getpgid(pid),signal.SIGKILL)
        except (ValueError,ProcessLookupError,PermissionError):pass
    SERVER_PIDFILE.unlink(missing_ok=True)
    SERVER_PROC=None
    # Let CUDA release buffers before re-launch on both T4s.
    time.sleep(2)

def server_command():
    c=PRODUCTION
    cmd=[str(SERVER),'--model',str(MODEL_PATH),'--alias',ALIAS,
         '--host','127.0.0.1','--port',str(LLAMA_PORT),
         '--split-mode','tensor','--main-gpu','0','--n-gpu-layers','all',
         '--tensor-split',c['tensor_split'],
         '--ctx-size',str(c['ctx']),
         '--batch-size',str(c['batch']),'--ubatch-size',str(c['ubatch']),
         '--flash-attn','on','--cache-type-k',c['kv'],'--cache-type-v',c['kv'],
         '--parallel','1','--cont-batching','--cache-prompt',
         '--jinja','--metrics','--perf','--no-webui','--no-context-shift',
         '--reasoning','off','--reasoning-budget','0',
         '--n-predict',str(MAX_OUTPUT_GUARD),
         '--mmproj',str(MMPROJ_PATH),
         '--image-min-tokens',str(c['image_min']),
         '--image-max-tokens',str(c['image_max']),
         '--spec-type','draft-mtp','--spec-draft-model',str(MTP_PATH),
         '--spec-draft-n-max',str(c['n']),
         '--spec-draft-p-min',str(c['pmin'])]
    # No auto-fit: the requested 64K must not be reduced to make the model fit.
    if CAPS.get('fit'): cmd+=['--fit','off']
    return cmd

def start_backend(timeout=600):
    global SERVER_PROC
    kill_server()
    env=os.environ.copy()
    env['LLAMA_API_KEY']=LLAMA_API_KEY
    env.pop('GGML_CUDA_P2P',None)  # PCIe T4 pair: no verified peer access.
    env.pop('CUDA_SCALE_LAUNCH_QUEUES',None)
    env['NCCL_DEBUG']='WARN'
    env['GGML_CUDA_PEER_MAX_BATCH_SIZE']='512'
    SERVER_LOG.write_text('')
    logf=open(SERVER_LOG,'a',buffering=1)
    try: SERVER_PROC=subprocess.Popen(server_command(),stdout=logf,
                     stderr=subprocess.STDOUT,env=env,start_new_session=True)
    finally:logf.close()
    SERVER_PIDFILE.write_text(str(SERVER_PROC.pid))
    t0=time.monotonic()
    while time.monotonic()-t0<timeout:
        if SERVER_PROC.poll() is not None:
            raise RuntimeError('llama-server exited at startup; log tail:\n'+SERVER_LOG.read_text(errors='ignore')[-5000:])
        if backend_health():
            return time.monotonic()-t0
        time.sleep(2)
    raise TimeoutError('llama-server failed to become healthy; log tail:\n'+SERVER_LOG.read_text(errors='ignore')[-5000:])


## 5 · Start production inference server

In [ ]:
# These are startup checks, not inference benchmarks. Never reduce the requested 64K.
assert PRODUCTION['ctx'] == 65536, 'Production context must remain exactly 65,536 tokens.'
start_cmd = server_command()
assert '--ctx-size' in start_cmd and start_cmd[start_cmd.index('--ctx-size') + 1] == '65536'
secs = start_backend()

# Verify llama-server's reported slot capacity where supported by its logs.
startup_log = SERVER_LOG.read_text(errors='replace')
ctx_reports = re.findall(r'\bn_ctx_slot\s*=\s*(\d+)\b', startup_log)
if not ctx_reports:
    ctx_reports = re.findall(r'\bn_ctx\s*=\s*(\d+)\b', startup_log)
if ctx_reports:
    allocated_ctx = int(ctx_reports[-1])
    if allocated_ctx != 65536:
        kill_server()
        raise RuntimeError(f'64K requirement not met: server reports {allocated_ctx:,} tokens.')
    print(f'Confirmed server context allocation: {allocated_ctx:,} tokens.')
else:
    print('NOTICE: llama-server log did not report a parseable slot context; requested --ctx-size 65536 is verified in the launch command. Inspect the server log for the allocation size.')

# Headroom is informational only, never automatically changes the context or profile.
try:
    snapshot = subprocess.check_output([
        'nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'
    ], text=True, timeout=15)
    remaining = [int(line.strip()) for line in snapshot.splitlines() if line.strip().isdigit()]
    if remaining:
        print('GPU free memory after startup:', ', '.join(f'{m:,} MiB' for m in remaining))
        if min(remaining) < 768:
            print('WARNING: Low free GPU memory; large prompts may cause OOM at 64K.')
except (OSError, subprocess.SubprocessError, ValueError) as e:
    print(f'GPU memory report unavailable: {type(e).__name__}')

DEPLOY = PRODUCTION.copy()
print(f"64K PRODUCTION READY in {secs:.1f}s | {ALIAS} | {DEPLOY['ctx']:,} context | GPU vision | MTP-2 p=0.40")
print(f'Local backend: http://127.0.0.1:{LLAMA_PORT} (not publicly exposed)')
print('No benchmark prompts or synthetic-image tests were executed.')


## 6 · OpenAI / Anthropic API gateway (automatically generated key)

In [ ]:
import threading, uvicorn
# Terminate a previous file-backed proxy from an earlier notebook session if present.
stop_pidfile(STATE_DIR/'openai-guard.pid')
os.environ['LLAMA_BACKEND'] = f'http://127.0.0.1:{LLAMA_PORT}'
os.environ['LLAMA_API_KEY'] = LLAMA_API_KEY
os.environ['LLAMA_MODEL_ALIAS'] = ALIAS
os.environ['MAX_OUTPUT_GUARD'] = str(MAX_OUTPUT_GUARD)
os.environ['MAX_ACTIVE_REQUESTS'] = '2'
os.environ['ALLOW_MODEL_ALIAS_REDIRECT'] = '0'


In [ ]:
"""Authenticated dual-format gateway for an on-host llama.cpp server.

Serves OpenAI Chat/Responses and Anthropic Messages from the SAME GGUF.
Use the generated key over a temporary Cloudflare HTTPS URL, and keep
llama.cpp itself bound to 127.0.0.1. Do NOT enable llama-server agent tools.

Environment:
  LLAMA_API_KEY                 generated automatically in notebook cell 1
  LLAMA_BACKEND                 default http://127.0.0.1:8081
  LLAMA_MODEL_ALIAS             default qwen3.8-27b-official
  MAX_OUTPUT_GUARD              default 8192
  MAX_JSON_BYTES                default 16777216
  MAX_ACTIVE_REQUESTS           default 4
  ALLOW_MODEL_ALIAS_REDIRECT    default 0, see explanation below
"""
import asyncio
import hmac
import json
import os

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, Response, StreamingResponse

BACKEND = os.environ.get('LLAMA_BACKEND', 'http://127.0.0.1:8081').rstrip('/')
TOKEN = os.environ.get('LLAMA_API_KEY', '')
if not TOKEN:
    raise RuntimeError('LLAMA_API_KEY is required; refusing to start an unauthenticated gateway')
MODEL_ALIAS = os.environ.get('LLAMA_MODEL_ALIAS', 'qwen3.8-27b-official')
MAX_OUT = int(os.environ.get('MAX_OUTPUT_GUARD', '8192'))
MAX_BYTES = int(os.environ.get('MAX_JSON_BYTES', str(16 * 1024 * 1024)))
MAX_ACTIVE = max(1, int(os.environ.get('MAX_ACTIVE_REQUESTS', '4')))
ALLOW_MODEL_ALIAS_REDIRECT = os.environ.get('ALLOW_MODEL_ALIAS_REDIRECT', '0') == '1'

app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
client = httpx.AsyncClient(timeout=httpx.Timeout(960.0, connect=15.0),
                           limits=httpx.Limits(max_connections=10, max_keepalive_connections=5))
slots = asyncio.Semaphore(MAX_ACTIVE)

# Only inference paths; importantly no /slots, /props, /lora-adapters, /metrics,
# /apply-template, file-ops, or arbitrary paths from the model server.
ALLOWED = {
    ('GET', 'models'),
    ('POST', 'chat/completions'),
    ('POST', 'completions'),
    ('POST', 'responses'),
    ('POST', 'messages'),
    ('POST', 'messages/count_tokens'),
}
# Forward protocol extension headers intact; NEVER forward caller credentials to
# the backend. We set the backend credential explicitly after validation.
FORWARD_REQUEST_HEADERS = {
    'accept', 'content-type', 'anthropic-version', 'anthropic-beta',
    'anthropic-dangerous-direct-browser-access', 'x-request-id',
}
FORWARD_RESPONSE_HEADERS = {
    'content-type', 'cache-control', 'x-request-id', 'retry-after',
    'anthropic-ratelimit-requests-remaining', 'anthropic-ratelimit-tokens-remaining',
}


def permitted(request: Request) -> bool:
    auth = request.headers.get('authorization', '')
    if auth.lower().startswith('bearer '):
        candidate = auth[7:].strip()
        if hmac.compare_digest(candidate, TOKEN):
            return True
    # Claude Code can use ANTHROPIC_API_KEY (x-api-key) instead of bearer.
    key = request.headers.get('x-api-key', '')
    return bool(key) and hmac.compare_digest(key, TOKEN)


def error(message: str, status: int, anthropic: bool = False):
    if anthropic:
        return JSONResponse({'type': 'error', 'error': {'type': 'invalid_request_error', 'message': message}},
                            status_code=status)
    return JSONResponse({'error': {'message': message, 'type': 'invalid_request_error'}}, status_code=status)


@app.get('/health')
async def health():
    # Does not expose stack, token, hardware, nor private model path.
    return {'status': 'ok'}


@app.api_route('/v1/{path:path}', methods=['GET', 'POST'])
async def proxy(path: str, request: Request):
    anthropic = path.startswith('messages')
    if not permitted(request):
        return error('Unauthorized', 401, anthropic)
    if (request.method, path) not in ALLOWED:
        return error('Unsupported endpoint', 404, anthropic)
    cl = request.headers.get('content-length')
    if cl is not None and cl.isdecimal() and int(cl) > MAX_BYTES:
        return error('Request body too large', 413, anthropic)
    # Do not rely on Content-Length, which a client may omit or falsify.
    chunks, consumed = [], 0
    async for chunk in request.stream():
        consumed += len(chunk)
        if consumed > MAX_BYTES:
            return error('Request body too large', 413, anthropic)
        chunks.append(chunk)
    body = b''.join(chunks)
    if request.method == 'POST':
        try:
            data = json.loads(body)
            if not isinstance(data, dict):
                return error('Expected a JSON object', 400, anthropic)
        except (json.JSONDecodeError, UnicodeDecodeError):
            return error('Invalid JSON', 400, anthropic)
        # One alias describes the real loaded Qwen model. Claude Code should set
        # ANTHROPIC_MODEL and ANTHROPIC_DEFAULT_* to it. Optional redirect is
        # explicitly opt-in and does NOT change what model is actually used.
        requested = data.get('model')
        if (ALLOW_MODEL_ALIAS_REDIRECT and anthropic and isinstance(requested, str)
                and requested.startswith('claude-')):
            data['model'] = MODEL_ALIAS
        for field in ('max_tokens', 'max_completion_tokens', 'n_predict', 'max_output_tokens'):
            value = data.get(field)
            if type(value) is int and value > MAX_OUT:
                data[field] = MAX_OUT
        body = json.dumps(data, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    headers = {key: value for key, value in request.headers.items()
               if key.lower() in FORWARD_REQUEST_HEADERS}
    headers['Authorization'] = 'Bearer ' + TOKEN
    headers['Content-Type'] = 'application/json'
    try:
        # Limit requests on the server; 1 llama.cpp slot still serializes decoding.
        await asyncio.wait_for(slots.acquire(), timeout=20)
    except asyncio.TimeoutError:
        return error('Server busy; retry shortly', 429, anthropic)
    upstream = None
    try:
        upstream = await client.send(client.build_request(
            request.method, f'{BACKEND}/v1/{path}', content=body, headers=headers), stream=True)
        response_headers = {k: v for k, v in upstream.headers.items()
                            if k.lower() in FORWARD_RESPONSE_HEADERS}
        content_type = upstream.headers.get('content-type', '')
        if 'text/event-stream' in content_type.lower():
            async def generate():
                try:
                    async for part in upstream.aiter_raw():
                        yield part
                finally:
                    await upstream.aclose()
                    slots.release()
            # StreamingResponse will preserve raw SSE event boundaries and payloads.
            return StreamingResponse(generate(), status_code=upstream.status_code,
                                     headers=response_headers, media_type='text/event-stream')
        content = await upstream.aread()
        await upstream.aclose()
        slots.release()
        return Response(content=content, status_code=upstream.status_code,
                        headers=response_headers)
    except httpx.TimeoutException:
        if upstream is not None:
            await upstream.aclose()
        slots.release()
        return error('Upstream timeout', 504, anthropic)
    except (httpx.HTTPError, OSError):
        if upstream is not None:
            await upstream.aclose()
        slots.release()
        return error('Upstream unavailable', 502, anthropic)

In [ ]:
if 'inline_gateway_server' in globals():
    inline_gateway_server.should_exit=True
    if 'inline_gateway_thread' in globals(): inline_gateway_thread.join(timeout=12)
inline_gateway_server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0',port=PORT,log_level='warning',access_log=False))
inline_gateway_thread = threading.Thread(target=inline_gateway_server.run,daemon=True,name='qwen38_inline_gateway')
inline_gateway_thread.start()
for _ in range(60):
    if local_health():break
    time.sleep(.5)
assert local_health(), 'Inline gateway did not start; check port conflicts or kernel logs'
print('Inline API gateway ready: http://127.0.0.1:%s/v1'%(PORT))


## 7 · Automatically generate a Cloudflare Quick Tunnel URL

No account, named tunnel, token file or DNS setup. The URL changes on restart. Cloudflare Quick Tunnels do not support SSE (`stream: true`); use non-streaming requests.

In [ ]:
# Install cloudflared if needed. A pre-attached suitable binary can be reused.
if RUN_PUBLIC_TUNNEL:
    if not CLOUDFLARED.exists():
        print('Downloading cloudflared (public GitHub release; no login).')
        subprocess.run([
            'curl', '-fL', '--retry', '3', '--connect-timeout', '20',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-o', str(CLOUDFLARED)
        ], check=True)
    CLOUDFLARED.chmod(0o755)
    print('Cloudflared:', subprocess.check_output([str(CLOUDFLARED),'--version'],text=True).strip())
else:
    print('Cloudflare tunnel disabled; local API only.')

In [ ]:
# No Cloudflare secret, token file or preconfigured domain.
# Capture the randomly generated trycloudflare.com address from cloudflared logs.
CF_PIDFILE = STATE_DIR / 'cloudflared.pid'
CF_LOG = LOG_DIR / 'cloudflared.log'
CF_HOME = STATE_DIR / 'isolated-cloudflared-home'
CF_HOME.mkdir(parents=True, exist_ok=True)
cf_proc = None
QUICK_URL_RE = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com', re.I)

def start_quick_tunnel(timeout=90):
    global cf_proc, PUBLIC_BASE_URL
    if not RUN_PUBLIC_TUNNEL:
        PUBLIC_BASE_URL = None
        return None
    stop_pidfile(CF_PIDFILE)
    PUBLIC_BASE_URL = None
    CF_LOG.write_text('')
    env = os.environ.copy()
    # Prevent a ~/.cloudflared/config.yml or a preexisting tunnel credential
    # from interfering with the zero-config Quick Tunnel setup.
    env['HOME'] = str(CF_HOME)
    env['XDG_CONFIG_HOME'] = str(CF_HOME)
    for key in ('TUNNEL_TOKEN', 'TUNNEL_ORIGIN_CERT', 'TUNNEL_CRED_FILE'):
        env.pop(key, None)
    cmd = [str(CLOUDFLARED), 'tunnel', '--no-autoupdate',
           '--protocol', 'http2', '--loglevel', 'info',
           '--url', f'http://127.0.0.1:{PORT}']
    with CF_LOG.open('a', buffering=1) as logf:
        cf_proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT,
                                   env=env, start_new_session=True)
    CF_PIDFILE.write_text(str(cf_proc.pid))
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        content = CF_LOG.read_text(errors='replace')
        match = QUICK_URL_RE.search(content)
        if cf_proc.poll() is not None:
            raise RuntimeError('Cloudflare Quick Tunnel exited during setup:\n' + content[-5000:])
        if match:
            PUBLIC_BASE_URL = match.group(0).rstrip('/') + '/v1'
            (STATE_DIR / 'public_endpoint.json').write_text(json.dumps({
                'base_url': PUBLIC_BASE_URL, 'model': ALIAS,
                'context': PRODUCTION['ctx'], 'key_generated_locally': True
            }, indent=2))
            print('Temporary Cloudflare API:', PUBLIC_BASE_URL)
            print('Cloudflare process PID:', cf_proc.pid)
            return PUBLIC_BASE_URL
        time.sleep(1)
    raise TimeoutError('Cloudflare did not provide a Quick Tunnel address. Log tail:\n' +
                       CF_LOG.read_text(errors='replace')[-5000:])

if RUN_PUBLIC_TUNNEL:
    start_quick_tunnel()
else:
    print(f'Local API: http://127.0.0.1:{PORT}/v1')

In [ ]:
# Readiness probe ONLY: no generated completion and no benchmark traffic.
def check_public_endpoint(attempts=12, delay=3):
    if not PUBLIC_BASE_URL:
        return False
    last_error = None
    for _ in range(attempts):
        try:
            response = requests.get(PUBLIC_BASE_URL.rstrip('/') + '/models',
                                    headers=auth_headers(), timeout=15)
            if response.status_code == 200:
                return True
            last_error = f'HTTP {response.status_code}: {response.text[:250]}'
        except requests.RequestException as exc:
            last_error = str(exc)[:250]
        time.sleep(delay)
    print('Public endpoint not ready yet:', last_error)
    return False

if RUN_PUBLIC_TUNNEL:
    if check_public_endpoint():
        print('PUBLIC API READY')
    else:
        print('Tunnel URL was created, but external readiness is not confirmed. Inspect:', CF_LOG)
else:
    print('Public tunnel disabled.')
print('\nClient settings')
print('Base URL  :', PUBLIC_BASE_URL or f'http://127.0.0.1:{PORT}/v1')
print('Model     :', ALIAS)
print('Context   :', DEPLOY['ctx'])
print('Max output:', MAX_OUTPUT_GUARD)
print('API key   :', LLAMA_API_KEY)
print('WARNING: Keep the auto-generated API key and saved notebook output private.')
print('Quick Tunnel clients: send stream=false (Cloudflare Quick Tunnels do not support SSE).')

## 8 · Optional process watchdog (no inference traffic)

In [ ]:
# Non-blocking infrastructure watchdog: no inference calls or benchmark traffic.
# It restarts only when a process exits; it never swaps to an unverified profile.
import threading
if '_PROD_WATCHDOG_STOP' in globals():
    _PROD_WATCHDOG_STOP.set()
_PROD_WATCHDOG_STOP=threading.Event()
def watchdog_loop(stop_event):
    failures=0
    while not stop_event.wait(30):
        global SERVER_PROC, cf_proc
        if SERVER_PROC is not None and SERVER_PROC.poll() is not None:
            failures+=1
            if failures>5:
                print('Watchdog stopped after five backend crashes; inspect',SERVER_LOG)
                return
            print(f'Backend exited. Restarting fixed production profile ({failures}/5)...')
            try:
                start_backend(timeout=600)
                print('Backend restarted successfully.')
            except Exception as exc:
                print('Backend restart failed:',str(exc)[:500])
        if RUN_PUBLIC_TUNNEL and 'cf_proc' in globals() and cf_proc.poll() is not None:
            print('Quick Tunnel exited; generating a new temporary URL...')
            try:
                start_quick_tunnel()
                if check_public_endpoint(attempts=4, delay=2):
                    print('New temporary endpoint is reachable:', PUBLIC_BASE_URL)
                else:
                    print('New endpoint created but readiness not confirmed:', PUBLIC_BASE_URL)
            except Exception as exc:
                print('Quick Tunnel restart failed:', str(exc)[:500])

watchdog_thread=threading.Thread(target=watchdog_loop,args=(_PROD_WATCHDOG_STOP,),
                                daemon=True,name='qwen38_production_watchdog')
watchdog_thread.start()
print('Production process watchdog started (no performance tests).')

## API usage, security and limitations

**Model:** `qwen3.8-27b-official` · **Context requested:** 65,536 tokens · **Max output:** 8,192 tokens. Run all Kaggle cells sequentially with Internet enabled and 2× T4 GPUs.

- **Hugging Face:** Public `ggml-org/Qwen3.8-27B-GGUF` base GGUF, MTP GGUF and vision projector are fetched anonymously with `token=False`. Existing validated files are reused. No `HF_TOKEN`.
- **Cloudflare:** `cloudflared tunnel --url http://127.0.0.1:8080` generates a different public HTTPS URL each time. No Cloudflare account, `CF_TUNNEL_TOKEN`, custom domain, or preconfigured tunnel. The latest URL is printed and also written (WITHOUT the key) to `state/public_endpoint.json`.
- **Client authorization:** A random key is created with Python `secrets.token_urlsafe(32)` on startup, not read from Kaggle Secrets. Every public request still needs `Authorization: Bearer <generated key>`; Anthropic clients may use `x-api-key`. Treat the generated key as private; avoid publishing outputs containing it. A restart of the Kaggle kernel generates a new key.
- **Endpoints:** `GET /v1/models`, `POST /v1/chat/completions`, `POST /v1/completions`, `POST /v1/responses`, `POST /v1/messages`, `POST /v1/messages/count_tokens`. Vision uses `image_url` content in OpenAI chat messages.
- **Non-streaming example:**

```python
import requests
response = requests.post(
    PUBLIC_BASE_URL + '/chat/completions',
    headers={'Authorization': 'Bearer ' + LLAMA_API_KEY},
    json={'model': ALIAS, 'messages': [{'role': 'user', 'content': 'Hello'}], 'stream': False},
    timeout=960,
)
print(response.status_code, response.text[:2000])
```

**Important:** Cloudflare Quick Tunnels are for testing/development, do not guarantee uptime, and **do not support SSE streaming**. Prefer `stream: false` from external clients. The endpoint disappears when its `cloudflared` process or Kaggle runtime stops; restarting creates a new hostname. The backend listens only on `127.0.0.1`, and the API gateway allowlists inference routes. Temporary public URLs still require careful access control; the generated API key is what protects inference from unauthorized callers. Kaggle Internet and GPU runtime limits still apply.

**64K VRAM caution:** 65,536 tokens were explicitly requested but NOT proven to fit the assigned dual-T4 hardware. The notebook will fail startup instead of silently reducing context. Inference performance, Cloudflare proxy timeouts and model availability vary by session.